# 02 — Validação do HBase

Notebook destinado à validação da camada de armazenamento NoSQL em tempo real da pipeline de e-commerce. São verificadas as tabelas utilizadas pelo processamento streaming, suas famílias de colunas, dados de métricas, alertas e eventos em tempo real.

**Tabelas analisadas:** `streaming_metrics`, `streaming_alerts` e `event_realtime`.

## 1. Configuração

Os parâmetros abaixo permitem executar o notebook em uma instalação local ou em um ambiente configurado por variáveis de ambiente.

In [ ]:
import os
import subprocess
import json
from pathlib import Path

HBASE_SHELL = os.getenv("HBASE_SHELL", "hbase shell")
HBASE_NAMESPACE = os.getenv("HBASE_NAMESPACE", "default")

TABLES = [
    "streaming_metrics",
    "streaming_alerts",
    "event_realtime",
]

print(f"Namespace: {HBASE_NAMESPACE}")
print("Tabelas:")
for table in TABLES:
    print(f"- {table}")

## 2. Verificação da configuração HBase

A estrutura do projeto deve conter os arquivos de configuração e os scripts de schema utilizados para criar a camada HBase.

In [ ]:
project_root = Path.cwd()

config_files = [
    project_root / "configs/hbase/hbase-site.xml",
    project_root / "configs/hbase/hbase-env.sh",
    project_root / "sql/hbase/schema.hbase",
    project_root / "sql/hbase/create_tables.hbase",
    project_root / "sql/hbase/seed_data.hbase",
]

for path in config_files:
    print(f"{path}: {'OK' if path.exists() else 'NÃO ENCONTRADO'}")

## 3. Estrutura definida para as tabelas

As famílias abaixo seguem o schema HBase utilizado pela pipeline.

In [ ]:
expected_families = {
    "streaming_metrics": {"metrics", "metadata"},
    "streaming_alerts": {"alert", "event", "metadata"},
    "event_realtime": {"event", "entity", "metadata"},
}

for table, families in expected_families.items():
    print(f"{table}: {', '.join(sorted(families))}")

## 4. Conexão com o HBase Shell

Esta função executa comandos do HBase Shell. Se o HBase não estiver iniciado, as células seguintes informarão o erro de conexão sem impedir a análise dos arquivos de schema.

In [ ]:
def hbase_command(command):
    result = subprocess.run(
        ["bash", "-lc", f"echo {json.dumps(command)} | hbase shell -n"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or "Falha no HBase Shell")
    return result.stdout


hbase_available = True
try:
    status_output = hbase_command("status")
    print(status_output)
except Exception as exc:
    hbase_available = False
    print(f"HBase indisponível no momento: {exc}")

## 5. Existência das tabelas

Verifica se as três tabelas previstas pela arquitetura estão criadas no HBase.

In [ ]:
table_status = {}

if hbase_available:
    output = hbase_command("list")
    for table in TABLES:
        exists = any(line.strip() == table for line in output.splitlines())
        table_status[table] = exists
        print(f"{table}: {'OK' if exists else 'NÃO ENCONTRADA'}")
else:
    print("Validação remota ignorada porque o HBase Shell não está disponível.")

## 6. Inspeção dos schemas

Consulta a descrição das tabelas para verificar as famílias de colunas efetivamente registradas.

In [ ]:
if hbase_available:
    for table in TABLES:
        print(f"\n===== {table} =====")
        try:
            print(hbase_command(f"describe '{table}'"))
        except Exception as exc:
            print(f"Não foi possível descrever {table}: {exc}")
else:
    print("A descrição das tabelas será executada quando o HBase estiver disponível.")

## 7. Validação das métricas de streaming

A tabela `streaming_metrics` recebe resultados agregados produzidos pelo processamento em tempo real, como contagem de eventos e valor da métrica por janela.

In [ ]:
if hbase_available:
    try:
        metrics_output = hbase_command("scan 'streaming_metrics', {LIMIT => 20}" )
        print(metrics_output)
    except Exception as exc:
        print(f"Não foi possível consultar streaming_metrics: {exc}")
else:
    print("Tabela streaming_metrics disponível para consulta após inicialização do HBase.")

## 8. Validação dos alertas

A tabela `streaming_alerts` mantém alertas detectados pelo processamento streaming, incluindo tipo, severidade, mensagem e referência ao evento.

In [ ]:
if hbase_available:
    try:
        alerts_output = hbase_command("scan 'streaming_alerts', {LIMIT => 20}" )
        print(alerts_output)
    except Exception as exc:
        print(f"Não foi possível consultar streaming_alerts: {exc}")
else:
    print("Tabela streaming_alerts disponível para consulta após inicialização do HBase.")

## 9. Validação dos eventos em tempo real

A tabela `event_realtime` permite consultar eventos individualmente por sua row key, mantendo atributos do evento e metadados.

In [ ]:
if hbase_available:
    try:
        events_output = hbase_command("scan 'event_realtime', {LIMIT => 20}" )
        print(events_output)
    except Exception as exc:
        print(f"Não foi possível consultar event_realtime: {exc}")
else:
    print("Tabela event_realtime disponível para consulta após inicialização do HBase.")

## 10. Validação dos dados de seed

O arquivo `seed_data.hbase` contém registros controlados para validar a leitura das três tabelas antes da execução contínua da pipeline.

In [ ]:
seed_file = project_root / "sql/hbase/seed_data.hbase"

if seed_file.exists():
    seed_text = seed_file.read_text(encoding="utf-8")
    for table in TABLES:
        print(f"{table}: {'OK' if table in seed_text else 'NÃO ENCONTRADA'}")
else:
    print("Arquivo de seed não encontrado.")

## 11. Consulta por row key

As row keys utilizadas nos dados de validação permitem demonstrar consultas direcionadas, evitando depender exclusivamente de scans completos.

In [ ]:
row_keys = {
    "streaming_metrics": "CLICK#2026-09-12T23:00:00Z#product-001",
    "streaming_alerts": "ALERT#2026-09-12T23:01:00Z#alert-001",
    "event_realtime": "CLICK#event-000001",
}

if hbase_available:
    for table, row_key in row_keys.items():
        print(f"\n===== {table} / {row_key} =====")
        try:
            print(hbase_command(f"get '{table}', '{row_key}'"))
        except Exception as exc:
            print(f"Não foi possível consultar a row key: {exc}")
else:
    print("Consultas por row key serão executadas com o HBase ativo.")

## 12. Resultado da validação

A camada HBase é responsável pela persistência de baixa latência dos resultados do processamento streaming. As tabelas `streaming_metrics` e `streaming_alerts` recebem resultados derivados das janelas e regras de alerta do Flink, enquanto `event_realtime` mantém eventos relevantes para consulta imediata.

Com essa estrutura, o HBase complementa o HDFS: o HDFS mantém o histórico distribuído e o HBase atende aos padrões de acesso em tempo real.